In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

In [ ]:
N_Colores = 16

def cargar_imagen(ruta):
    img = Image.open(ruta)
    if img.mode != 'RGB':
        img = img.convert('RGB')
    return np.array(img)

In [ ]:
ruta_imagen = 'img/pensamientos.jpg'
imagen_original = cargar_imagen(ruta_imagen)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(imagen_original[:, :, 0], cmap='Reds')
axes[1].imshow(imagen_original[:, :, 1], cmap='Greens')
axes[2].imshow(imagen_original[:, :, 2], cmap='Blues')

axes[0].set_title('Canal Rojo')
axes[1].set_title('Canal Verde')
axes[2].set_title('Canal Azul')

plt.show()


In [ ]:
# Paso 1: Cuantización de colores
def cuantizar_colores(imagen, n_colores):
    h, w, c = imagen.shape

    pixels = imagen.reshape(-1, 3)
    pixels_float = pixels.astype(float)

    kmeans = KMeans(
        n_clusters=n_colores,
        init='k-means++',
        max_iter=300,
        n_init=10,
        random_state=42,
        verbose=0
    )

    labels = kmeans.fit_predict(pixels_float)
    colores_paleta = kmeans.cluster_centers_

    imagen_cuantizada = colores_paleta[labels]

    imagen_cuantizada = imagen_cuantizada.reshape(h, w, c)

    imagen_cuantizada = imagen_cuantizada.astype(np.uint8)
    colores_paleta = colores_paleta.astype(np.uint8)

    return imagen_cuantizada, colores_paleta, labels.reshape(h, w)

In [ ]:
imagen_cuantizada, colores_paleta, labels = cuantizar_colores(imagen_original, N_Colores)


In [ ]:
img_pil = Image.fromarray(imagen_cuantizada)
img_pil.save('output/paso_1_colores_cuantizados.png')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].imshow(imagen_original)
axes[1].imshow(imagen_cuantizada)

plt.show()

In [ ]:
from scipy import ndimage
from skimage import measure
import pandas as pd

In [ ]:
# Paso 2: segmentar regiones

def segmentar_regiones(imagen_cuantizada, labels, colores_paleta):
    h, w = labels.shape

    mapa_regiones = np.zeros((h, w), dtype=np.int32)

    regiones_info = []
    region_id_global = 1

    for color_idx in range(len(colores_paleta)):
        mascara_color = (labels == color_idx).astype(np.uint8)
        
        regiones_etiquetadas, num_regiones = ndimage.label(mascara_color)

        if num_regiones == 0:
            continue

        for region_idx in range(1, num_regiones + 1):
            mascara_region = (regiones_etiquetadas == region_idx)
            area = np.sum(mascara_region)

            mapa_regiones[mascara_region] = region_id_global

            regiones_info.append({
                'region_id': region_id_global,
                'color_idx': color_idx,
                'color_rgb': tuple(colores_paleta[color_idx]),
                'area_pixels': area,
                'porcentaje': 100 * area / (h * w)
            })

            region_id_global += 1

    df_regiones = pd.DataFrame(regiones_info)
    
    return mapa_regiones, df_regiones

def visualizar_regiones(imagen_original, imagen_cuantizada, mapa_regiones, df_regiones):
    """Visualiza las regiones segmentadas"""
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 14))
    
    # Imagen original
    axes[0, 0].imshow(imagen_original)
    axes[0, 0].set_title('Imagen Original', fontsize=14, fontweight='bold')
    axes[0, 0].axis('off')
    
    # Imagen cuantizada
    axes[0, 1].imshow(imagen_cuantizada)
    axes[0, 1].set_title(f'Imagen Cuantizada ({N_Colores} colores)', fontsize=14, fontweight='bold')
    axes[0, 1].axis('off')
    
    # Mapa de regiones (cada región con color aleatorio para distinguir)
    # Usar colormap para visualizar mejor
    regiones_visuales = mapa_regiones.copy()
    axes[1, 0].imshow(regiones_visuales, cmap='nipy_spectral', interpolation='nearest')
    axes[1, 0].set_title(f'Regiones Segmentadas ({len(df_regiones)} regiones)', 
                         fontsize=14, fontweight='bold')
    axes[1, 0].axis('off')
    
    
    # Imagen cuantizada con bordes de regiones
    from skimage.segmentation import find_boundaries
    bordes = find_boundaries(mapa_regiones, mode='thick')
    
    img_con_bordes = imagen_cuantizada.copy()
    img_con_bordes[bordes] = [0, 0, 0]  # Bordes negros
    
    axes[1, 1].imshow(img_con_bordes)
    axes[1, 1].set_title('Regiones con Bordes Delimitados', fontsize=14, fontweight='bold')
    axes[1, 1].axis('off')
    
    plt.tight_layout()
    plt.savefig('output/paso_2_regiones_segmentadas.png')
    plt.show()


In [ ]:
mapa_regiones, df_regiones = segmentar_regiones(imagen_cuantizada, labels, colores_paleta)

In [ ]:
visualizar_regiones(imagen_original, imagen_cuantizada, mapa_regiones, df_regiones)

In [ ]:
df_regiones[df_regiones['area_pixels'] == 1].describe()

In [ ]:
from scipy.ndimage import distance_transform_edt
from collections import defaultdict

In [ ]:

def encontrar_vecino_mas_cercano(mapa_regiones, region_id, mascara_region, df_regiones):
    """
    Encuentra la región vecina más grande (preferiblemente del mismo color)
    
    Args:
        mapa_regiones: mapa de regiones actual
        region_id: ID de la región a fusionar
        mascara_region: máscara binaria de la región
        df_regiones: DataFrame con info de regiones
    
    Returns:
        ID de la región vecina más apropiada
    """
    # Dilatar la región para encontrar vecinos
    from scipy.ndimage import binary_dilation
    
    estructura = np.ones((3, 3))  # 8-connectivity
    mascara_dilatada = binary_dilation(mascara_region, structure=estructura)
    
    # Encontrar regiones vecinas (en el borde)
    borde = mascara_dilatada & ~mascara_region
    regiones_vecinas = np.unique(mapa_regiones[borde])
    regiones_vecinas = regiones_vecinas[regiones_vecinas != 0]  # Excluir background
    regiones_vecinas = regiones_vecinas[regiones_vecinas != region_id]  # Excluir a sí misma
    
    if len(regiones_vecinas) == 0:
        return None
    
    # Obtener info de la región actual
    color_actual = df_regiones[df_regiones['region_id'] == region_id]['color_id'].values[0]
    
    # Priorizar vecinos del mismo color, luego por tamaño
    vecinos_info = df_regiones[df_regiones['region_id'].isin(regiones_vecinas)].copy()
    
    # Score: mismo color = +1000000, luego por área
    vecinos_info['score'] = vecinos_info['area_pixels'].copy()
    vecinos_info.loc[vecinos_info['color_id'] == color_actual, 'score'] += 1000000
    
    mejor_vecino = vecinos_info.loc[vecinos_info['score'].idxmax(), 'region_id']
    
    return mejor_vecino